# 09 — OOF Hill Climbing Ensemble and submission

Notebook 08が保存した3モデルのOOF/test予測を統合し、最終submissionを作ります。

Hill Climbingのweightを学習した行でそのまま評価すると楽観的になるため、各outer foldをhold-outし、
残り4 foldだけでblendを探索します。test予測は5個のfold別blend予測を平均します。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import Baseline
from ensemble import cross_fitted_hill_climb, load_aligned_predictions
from hpo import HPO_MODELS, model_slug

TARGET = Baseline.TARGET
ID_COLUMN = Baseline.ID_COLUMN
FOLD_COLUMN = Baseline.FOLD_COLUMN
ARTIFACT_DIR = ROOT / "artifacts"
PREFIXES = {
    model_name: f"final_{model_slug(model_name)}"
    for model_name in HPO_MODELS
}

## 予測契約

3モデルのID、target、fold、test ID順が完全一致しない場合は処理を止めます。

In [ ]:
reference_oof, reference_test, oof_matrix, test_matrix = (
    load_aligned_predictions(
        ARTIFACT_DIR,
        PREFIXES,
        id_column=ID_COLUMN,
        target=TARGET,
        fold_column=FOLD_COLUMN,
    )
)
y_true = reference_oof[TARGET].to_numpy()
fold_ids = reference_oof[FOLD_COLUMN].to_numpy()
print("OOF matrix :", oof_matrix.shape)
print("test matrix:", test_matrix.shape)

## 単体性能と多様性

AUCが高くても相関がほぼ1の候補は、ensembleで追加価値を持ちにくい点に注意します。

In [ ]:
individual_scores = pd.DataFrame(
    {
        "model": HPO_MODELS,
        "oof_auc": [
            roc_auc_score(y_true, oof_matrix[:, index])
            for index in range(len(HPO_MODELS))
        ],
    }
).sort_values("oof_auc", ascending=False)
correlation = pd.DataFrame(oof_matrix, columns=HPO_MODELS).corr()
display(individual_scores)
display(correlation)

## Cross-fitted Hill Climbing

各roundで現在のblendを1%、2%、5%、10%、20%、35%、50%だけ各候補方向へ動かし、
fit側OOF AUCが改善する一手を採用します。改善がなくなった時点で停止します。

foldごとにweightを独立して学習するため、validation行のtargetはその行のblend weight決定に使われません。

In [ ]:
hill_result = cross_fitted_hill_climb(
    oof_matrix=oof_matrix,
    test_matrix=test_matrix,
    target=y_true,
    fold_ids=fold_ids,
    max_rounds=100,
    minimum_improvement=1e-7,
)

equal_oof = oof_matrix.mean(axis=1)
equal_test = test_matrix.mean(axis=1)
equal_auc = roc_auc_score(y_true, equal_oof)

best_single_row = individual_scores.iloc[0]
best_single_index = HPO_MODELS.index(best_single_row["model"])
best_single_auc = float(best_single_row["oof_auc"])
best_single_oof = oof_matrix[:, best_single_index]
best_single_test = test_matrix[:, best_single_index]

method_scores = pd.DataFrame(
    [
        {"method": "best_single", "oof_auc": best_single_auc},
        {"method": "equal_weight", "oof_auc": equal_auc},
        {
            "method": "cross_fitted_hill_climb",
            "oof_auc": hill_result["oof_auc"],
        },
    ]
).sort_values("oof_auc", ascending=False)
display(method_scores)

## weightの安定性

平均weightだけでなくfold間の標準偏差を確認します。foldごとにweightが大きく変わる場合、
Hill Climbingは不安定です。その場合は少しのAUC改善より単純平均・単体モデルを優先します。

In [ ]:
weights_by_fold = pd.DataFrame(
    hill_result["fold_weights"], columns=HPO_MODELS
)
weights_by_fold.index.name = "held_out_fold"
weight_summary = pd.DataFrame(
    {
        "model": HPO_MODELS,
        "mean_weight": weights_by_fold.mean(axis=0).to_numpy(),
        "std_weight": weights_by_fold.std(axis=0, ddof=0).to_numpy(),
    }
).sort_values("mean_weight", ascending=False)
display(weights_by_fold)
display(weight_summary)

weight_summary.plot.bar(
    x="model", y="mean_weight", yerr="std_weight", legend=False
)
plt.ylabel("weight")
plt.title("Cross-fitted hill-climbing weights")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 保守的な最終選択

最高値を無条件採用せず、best singleより`5e-5`以上改善した場合だけensembleへ切り替えます。
ensemble同士ではAUCが高い方を使います。差が閾値未満なら、説明しやすいbest singleを残します。

In [ ]:
MINIMUM_ENSEMBLE_GAIN = 5e-5
ensemble_candidates = [
    ("equal_weight", equal_auc, equal_oof, equal_test),
    (
        "cross_fitted_hill_climb",
        hill_result["oof_auc"],
        hill_result["oof"],
        hill_result["test_pred"],
    ),
]
best_ensemble = max(ensemble_candidates, key=lambda item: item[1])

if best_ensemble[1] >= best_single_auc + MINIMUM_ENSEMBLE_GAIN:
    selected_method, selected_auc, final_oof, final_test = best_ensemble
else:
    selected_method = f"best_single:{best_single_row['model']}"
    selected_auc = best_single_auc
    final_oof = best_single_oof
    final_test = best_single_test

print("selected method:", selected_method)
print(f"selected OOF AUC: {selected_auc:.6f}")

## 最終submissionと監査ファイル

ID順、行数、確率範囲を検証してから保存します。

In [ ]:
submission = pd.DataFrame(
    {
        ID_COLUMN: reference_test[ID_COLUMN].to_numpy(),
        TARGET: final_test,
    }
)
assert len(submission) == len(reference_test)
assert submission[ID_COLUMN].equals(reference_test[ID_COLUMN])
assert submission[TARGET].between(0, 1).all()

submission_path = ROOT / "submission_final.csv"
submission.to_csv(submission_path, index=False)
pd.DataFrame(
    {
        ID_COLUMN: reference_oof[ID_COLUMN],
        TARGET: y_true,
        FOLD_COLUMN: fold_ids,
        "oof_prob": final_oof,
    }
).to_csv(ARTIFACT_DIR / "final_ensemble_oof.csv", index=False)
weights_by_fold.to_csv(
    ARTIFACT_DIR / "hill_climb_weights_by_fold.csv"
)
weight_summary.to_csv(
    ARTIFACT_DIR / "hill_climb_weight_summary.csv", index=False
)
hill_result["history"].to_csv(
    ARTIFACT_DIR / "hill_climb_history.csv", index=False
)
method_scores.to_csv(
    ARTIFACT_DIR / "final_ensemble_comparison.csv", index=False
)
pd.DataFrame(
    [{"selected_method": selected_method, "selected_oof_auc": selected_auc}]
).to_csv(ARTIFACT_DIR / "final_selection.csv", index=False)

print("saved:", submission_path)
print("shape:", submission.shape)
display(submission.head())

## 提出後

Public LBは記録しますが、見た後で特徴量・trial・weightを同じCVへ追加し続けると選択過適合が進みます。
次の実験を行う場合は、仮説、変更点、OOF差、fold別差、実行時間を先に記録し、
LBだけを理由に設定を戻したり足したりしないようにします。